# Mapping migration


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [ ]:
%store -r ecoreg_gdf occurrence_df data_dir

You will also need to import any libraries you are using in this notebook. In this notebook, you will need some additional packages to make interactive maps with vector data and access pre-defined month names.

In [ ]:
### import packages

### libraries for dynamic mapping

### manage coordinate ref systems
import cartopy.crs as ccrs

### customize dynamic plot
import panel as pn

### make dynamic plot with geodataframe
import hvplot.pandas
import holoviews as hv

### Run this fix if the plot is not showing, once per kernal session
hvplot.extension("bokeh")

### get month names
import calendar

### folder/file management
import os
import pathlib



## Step 5: Make an interactive map of GBIF observations

### Step 5A: Create a simplified `GeoDataFrame` for plotting

Plotting larger files can be time consuming. The code below will streamline plotting with `hvplot` by simplifying the geometry, projecting it to a Mercator projection that is compatible with `geoviews`, and cropping off areas in the Arctic:

1. Simplify the ecoregions with `.simplify(.05)`, and save it back to the `geometry` column.
2. Change the Coordinate Reference System (CRS) to Mercator with `.to_crs(ccrs.Mercator())`.
3. Use the plotting code that is already in the cell to check that the plotting runs quickly (less than a minute) and looks the way you want, making sure to change `gdf` to your `GeoDataFrame` name.

In [ ]:
### simplify the geometry to speed up processing
ecoreg_gdf.geometry = ecoreg_gdf.geometry.simplify(
    0.05, preserve_topology=False)

### change the CRS to Mercator for mapping
ecoreg_gdf = ecoreg_gdf.to_crs(ccrs.Mercator())

### check that the plot runs in a reasonable amount of time
ecoreg_gdf.hvplot(geo = True, crs = ccrs.Mercator())

### Step 5B: Map migration over time

1. If applicable, replace any variable names with the names you defined previously.
2. Replace `column_name_used_for_ecoregion_color` and `column_name_used_for_slider` with the column names you wish to use.
3. Customize your plot with your choice of title, tile source, color map, and size.

Your plot will probably still change months very slowly in your Jupyter notebook, because it calculates each month’s plot as needed. Open up the saved HTML file to see faster performance!

In [ ]:
### prep the data, then combine the occurrence data with the ecoregions

### rename columns in ecoregID to match occurrence_df
ecoreg_gdf = ecoreg_gdf.rename(columns = {'OBJECTID': 'ecoregion'})

### make ecoreg ID the index for merge with occurrence_df
### Run this once but comment out if you rerun
ecoreg_gdf = ecoreg_gdf.set_index('ecoregion')

### join the occurrences with the plotting GeoDataFrame
occurrence_gdf = ecoreg_gdf.join(occurrence_df)

occurrence_gdf


In [ ]:
### activate panel and hvplot extensions you'll need for the interactive plot
pn.extension()
hv.extension("bokeh")

In [ ]:
### plot the data

### get the plot bounds so they don't change with the slider
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month', 
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### plot occurrence by ecoregion and month
migration_plot = (
    occurrence_gdf
    .hvplot(
        ### Color by number veeries
        c = 'norm_occurrences',
        groupby = 'month',

        ### use background tiles
        geo = True, crs = ccrs.Mercator(), tiles = 'OSM',

        ### add a title
        title = "Yearly Veery Migration",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False, ## True will categorize the colorbar but then the scale changes with slide
        widgets = {'month': slider}
    )
)

### wrap  plot in the Panel layout
interactive_migration_plot = pn.panel(migration_plot)

### check it out
interactive_migration_plot

Due to some annoying issues with permissions for the basemap, we can't export the map you just made as an .html file with the basemap. So we'll make another version, without a basemap, to export. We'll revisit this for your portfolio assignment to make a nice map with a basemap.

In [ ]:

# make a path for your map to live

map_path = os.path.join(
    pathlib.Path.home(),
    ### point to folders where your data will live
    'Repos',
    'earth-data-portfolio',
    'coding-assignments',
    '02_migration',
    'figures',
    '04_veery_plot.html'
    )

### make the parent directory for the path
os.makedirs(os.path.dirname(map_path), exist_ok = True)
print(map_path)

In [ ]:
### make a version for export, without the basemap
export_migration_plot = (
    occurrence_gdf
    .hvplot(
        ### Color by number veeries
        c = 'norm_occurrences',
        groupby = 'month',

        ### add a title
        title = "Yearly Veery Migration",
        xlabel = "Longitude", ylabel = "Latitude",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'month': slider}
    )
)
export_migration_plot

In [ ]:
### export the map as an .html file
export_migration_plot.save(
    map_path,
    embed = True,
    resources = "inline",
)

print(f"Map saved to {map_path}")

### Reflect and Respond #2: Interpret your map
What seasonal trends do you see in the GBIF observations? How does your map align with the broader literature on Veery migration?

According to data compiles here from the [Global Biodiversity Information Facility](https://www.gbif.org/) Veeries overwinter in south America, particularly Brazil, and head north in the spring. From March-April they head up through Florida in and spread across the United states, expanding up into Canada through June, July, and August. Nearing the end of northern hemisphere summer they head south again stopping Cuba and central America on the way back to the Amazon Basin. Notably, some Veeries appear to in Ireland and the UK in September and October, though these observations, known as **vagrants**, are rare according to the [British Trust for Ornithology]( https://www.bto.org/learn/about-birds/birdfacts/veery) (Veery | BTO, n.d.).

This migration is in line with existing literature which describes migration (Heckscher et al., 2011). Though essentially all Veeries migrate not all individuals necessarily traverse the entire north American continent. A 2011 study tracked five individual Veeries whose summer breeding grounds were in Delaware (Heckscher et al., 2011). Of these five individuals two stopped at gulf islands, two stopped in central America, and one stopped both in central America and at gulf islands on the way north. Veeries in another 2015 study traveled further north traversing the Rocky Mountains to breed in the Okanagan region of British Columbia, Canada (Hobson & Kardynal, 2015).  This Okanagan population uses some of the same migratory flyways as another Pemberton Valley, BC population and the Delaware population (Kardynal & Hobson, 2017). Better geolocation technology has enabled scientists to parse population-level trends in south American wintering habitat among these unique populations (Kardynal & Hobson, 2017). Veeries use northern latitudes as breeding sites and migrate south to winter in warmer climates (Heckscher et al., 2015). Newer literature has begun identifying a second intra-tropical migration wherein Veeries move from the Central Brazil north into lowland Amazonia following a flood pulse, though the reason for this migration is still unclear (Heckscher et al., 2015). Veery migration is complex and databases like GBIF show exciting potential to learn more bout these emerging patterns.


##### References

<div style="padding-left: 0.5in; text-indent: -0.5in;">

<p>Heckscher, C. M., Halley, M. R., &amp; Stampul, P. M. (2015). Intratropical migration of a Nearctic-Neotropical migratory songbird (<em>Catharus fuscescens</em>) in South America with implications for migration theory. <em>Journal of Tropical Ecology, 31</em>(3), 285–289. <a href="https://doi.org/10.1017/S0266467415000024">https://doi.org/10.1017/S0266467415000024</a></p>

<p>Heckscher, C. M., Taylor, S. M., Fox, J. W., &amp; Afanasyev, V. (2011). Veery (<em>Catharus fuscescens</em>) wintering locations, migratory connectivity, and a revision of its winter range using geolocator technology. <em>The Auk, 128</em>(3), 531–542. <a href="https://doi.org/10.1525/auk.2011.10280">https://doi.org/10.1525/auk.2011.10280</a></p>

<p>Hobson, K. A., &amp; Kardynal, K. J. (2015). Western Veeries use an eastern shortest-distance pathway: New insights to migration routes and phenology using light-level geolocators. <em>The Auk, 132</em>(3), 540–550. <a href="https://doi.org/10.1642/AUK-14-260.1">https://doi.org/10.1642/AUK-14-260.1</a></p>

<p>Kardynal, K. J., &amp; Hobson, K. A. (2017). The pull of the Central Flyway? Veeries breeding in western Canada migrate using an ancestral eastern route. <em>Journal of Field Ornithology, 88</em>(3), 262–273. <a href="https://doi.org/10.1111/jofo.12207">https://doi.org/10.1111/jofo.12207</a></p>

<p><em>Veery</em>. (n.d.). BTO. Retrieved October 20, 2026, from <a href="https://www.bto.org/learn/about-birds/birdfacts/veery">https://www.bto.org/learn/about-birds/birdfacts/veery</a></p>

</div>